In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers

import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Dataset Inspection

The dataset contains chest X-ray images classified into three categories:

- COVID
- Normal
- Pneumonia

This section explores the dataset structure and class distribution before preprocessing and model development.

In [2]:
train_dir = "../data/train"
test_dir = "../data/test"

print("TRAIN CLASSES")
print(os.listdir(train_dir))

print("\nTEST CLASSES")
print(os.listdir(test_dir))

TRAIN CLASSES
['.DS_Store', 'Covid', 'Normal', 'Pneumonia']

TEST CLASSES
['.DS_Store', 'Covid', 'Normal', 'Pneumonia']


In [3]:
import os

train_dir = "../data/train"

classes = [
    folder
    for folder in os.listdir(train_dir)
    if not folder.startswith(".")
]

print("Classes:", classes)

Classes: ['Covid', 'Normal', 'Pneumonia']


In [4]:
for class_name in classes:
    path = os.path.join(train_dir, class_name)

    image_count = len([image for image in os.listdir(path)
            if not image.startswith(".")])

    print(f"{class_name}: {image_count}")


Covid: 111
Normal: 70
Pneumonia: 70


### Findings

The training dataset contains three classes:

| Class     | Images |
|-----------|--------|
| Covid     | 111    |
| Normal    | 70     |
| Pneumonia | 70     |

The dataset is relatively balanced, although the Covid class contains more images than the other two classes. The classification task is therefore a multi-class image classification problem with three target categories.

# 2. Data Preprocessing

Before training a neural network, image pixel values must be normalized.

Chest X-ray images contain pixel values ranging from 0 to 255. Neural networks generally perform better when inputs are scaled to a smaller range.

The ImageDataGenerator will therefore rescale all pixel values to the range 0–1 by dividing each pixel value by 255.

In [5]:
image_generator = ImageDataGenerator(rescale=1/255)

print("ImageDataGenerator created successfully.")

ImageDataGenerator created successfully.


### Preprocessing Strategy

For the initial model, only normalization is applied.

More advanced preprocessing techniques such as image augmentation, rotation, zooming, and horizontal flipping may be explored later when optimizing the convolutional neural network.


# 3. Create Training and Testing Generators

The ImageDataGenerator will now load images from the training and testing directories.

All images will be resized to a consistent size of 224 × 224 pixels before being passed into the neural network. Because this is a multi-class classification problem, categorical labels will be generated for each image.

In [9]:
train_generator = image_generator.flow_from_directory("../data/train",
    target_size=(224, 224), batch_size=32, class_mode="categorical")

test_generator = image_generator.flow_from_directory("../data/test",
    target_size=(224, 224), batch_size=32, class_mode="categorical",
    shuffle=False)

Found 251 images belonging to 3 classes.
Found 66 images belonging to 3 classes.


### Generator Verification

The training and testing generators were successfully created.

The class indices, training image count, and testing image count are inspected below to verify that the image loading pipeline is configured correctly.

In [10]:
print("Class Indices:")
print(train_generator.class_indices)

print("\nTraining Images:")
print(train_generator.samples)

print("\nTesting Images:")
print(test_generator.samples)

Class Indices:
{'Covid': 0, 'Normal': 1, 'Pneumonia': 2}

Training Images:
251

Testing Images:
66


### Findings

The data generators successfully loaded the dataset.

Class mappings:

| Class     | Label |
|---------==|-------|
| Covid     | 0     |
| Normal    | 1     |
| Pneumonia | 2     |

Dataset size:

| Dataset | Images |
|---------|--------|
| Training| 251    |
| Testing | 66     |

The image pipeline is correctly configured and ready for model development.

# 4. Baseline Neural Network

A simple baseline neural network will be created before developing a more sophisticated convolutional neural network (CNN).

The baseline model will:

- Flatten image data into a one-dimensional array
- Use a fully connected hidden layer
- Predict one of three classes using a softmax output layer

This baseline provides a performance benchmark that can later be compared against a CNN architecture.

In [11]:
baseline_model = Sequential([layers.Flatten(input_shape=(224, 224, 3)),
    layers.Dense(128, activation="relu"), layers.Dense(3, activation="softmax")])

baseline_model.summary()

c:\Users\wayne\Projects\covid-pneumonia-classifier\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 150528)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    19,267,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,268,099 (73.50 MB)

 Trainable params: 19,268,099 (73.50 MB)

 Non-trainable params: 0 (0.00 B)

### Model Compilation

The baseline model is compiled using the Adam optimizer and categorical cross-entropy loss function.

Because this is a multi-class classification problem involving three classes (Covid, Normal, and Pneumonia), categorical cross-entropy is used as the loss function. Model performance will be evaluated using classification accuracy.

In [12]:
baseline_model.compile(optimizer="adam", loss="categorical_crossentropy",
    metrics=["accuracy"])

print("Baseline model compiled successfully.")

Baseline model compiled successfully.


# 5. Baseline Model Training

The baseline neural network will now be trained using the training dataset and evaluated on the testing dataset.

The objective is not necessarily to achieve the highest possible accuracy but to establish a baseline performance benchmark that can later be compared to a convolutional neural network (CNN).

In [13]:
early_stop = EarlyStopping(monitor="val_loss", patience=3,
    restore_best_weights=True)

### Early Stopping

Early stopping is used to prevent excessive overfitting.

Training will automatically stop if validation loss fails to improve for several consecutive epochs.

In [14]:
history = baseline_model.fit(train_generator, validation_data=test_generator,
    epochs=20, callbacks=[early_stop])

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 806ms/step - accuracy: 0.4542 - loss: 75.5818 - val_accuracy: 0.4091 - val_loss: 28.1569
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 403ms/step - accuracy: 0.5657 - loss: 14.7439 - val_accuracy: 0.7727 - val_loss: 5.7488
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 377ms/step - accuracy: 0.7809 - loss: 4.7099 - val_accuracy: 0.6061 - val_loss: 10.8185
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 333ms/step - accuracy: 0.8645 - loss: 2.5423 - val_accuracy: 0.7879 - val_loss: 5.2094
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 317ms/step - accuracy: 0.8725 - loss: 2.2745 - val_accuracy: 0.7727 - val_loss: 4.6578
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 334ms/step - accuracy: 0.8924 - loss: 1.6974 - val_accuracy: 0.6212 - val_loss: 5.6169
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 355ms/step - accuracy: 0.9084 - loss: 1.3501 - val_accuracy: 0.8030 - val_loss: 3.6114
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 371ms/step - accuracy: 0.9243 - loss: 0.9880 - val_accuracy: 0.8182 - val_l

### Baseline Model Results

The baseline model achieved high training accuracy but showed clear signs of overfitting.

Although the model learned the training dataset well, validation loss remained significantly higher than training loss. This suggests that the flattened dense architecture does not generalize effectively to unseen chest X-ray images.

A convolutional neural network (CNN) is expected to improve performance by learning spatial features directly from the images while using a more suitable architecture for image classification.


In [15]:
baseline_loss, baseline_accuracy = baseline_model.evaluate(test_generator)

print(f"Baseline Test Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Test Loss: {baseline_loss:.4f}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8485 - loss: 1.8944 
Baseline Test Accuracy: 0.8485
Baseline Test Loss: 1.8944


### Baseline Model Evaluation

The baseline neural network achieved a test accuracy of approximately 84.9%.

Despite producing a strong classification accuracy, the model contains more than 19 million trainable parameters and exhibits signs of overfitting. The gap between training and testing performance suggests that the model has learned patterns specific to the training images rather than general image features.

A convolutional neural network (CNN) will now be developed to improve feature extraction and generalization while using an architecture better suited to image classification tasks.